# Citadel — CYMEK 500M CAMPAIGN (reserved final design)

Target: 500,000,000 consumed training tokens (Cymek
TrainingState.cumulative_tokens), milestones 50M/100M/200M/350M/500M.
Multi-session: every runtime ends with a durable recovery checkpoint; the
same CELL 1 resumes from the last committed generation.

BLOCKING prerequisite: docs/citadel/500M/CYMEK_REQUIRED_CHANGES.md items
B1 (materialized production corpus), B2 (production training entry point),
B3 (frozen tokenizer artifact). Until all three land and PRE500M goes
green, CELL 1 exits BLOCKED with the precise blocker list - by design.

T1D: EXECUTED/ARCHIVED. PRE50M: PASS. T1E: plan only. 5B corpus: not started.

In [ ]:
# CELL 0 - bootstrap: exact Citadel + exact canonical Cymek + campaign state
import os, subprocess, sys
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
os.environ['CITADEL_ROOT'] = repo
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
CAMPAIGN_STATE = '/content/drive/MyDrive/cymek_500m_state'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('drive mount unavailable:', exc)
if repo not in sys.path: sys.path.insert(0, repo)
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))
print('EXPECTED_CYMEK_SHA=28bf57a0d299a2c13a99fe0046616c00a1b8530c')
if rt_sha != '28bf57a0d299a2c13a99fe0046616c00a1b8530c':
    raise RuntimeError('CYMEK PIN MISMATCH: ' + rt_sha + ' - STOP')
SESSION = 'docs/citadel/tpu_receipts/cymek_500m'
print('SESSION_DIR=' + SESSION)

In [ ]:
# CELL 1 - RUN / RESUME CYMEK 500M CAMPAIGN
# Drives the audited Cymek production entry point through the 500M campaign
# orchestrator. Fail-closed until PRE500M is green. Same cell = fresh start
# or resume from the latest committed generation.
import importlib, json
from citadel_tpu import pre500m as _p500
from citadel_tpu import t1d_run as _t1d_module
from citadel_tpu import t1d_one_shot as _oshot
_oshot = importlib.reload(_oshot)
_p500 = importlib.reload(_p500)
t1d = importlib.reload(_t1d_module)
print('CYMEK 500M | orchestrator:', _oshot.ORCHESTRATOR_VERSION)
decision = _p500.build_next_500m_decision(**{})  # fail-closed probe
if not decision['ready_for_500m_training']:
    print('NEXT_500M_DECISION: BLOCKED')
    for b in decision['blocking_reasons']: print('  BLOCKER:', b)
    print('See docs/citadel/500M/CYMEK_REQUIRED_CHANGES.md - do not train on a blocked campaign.')
    raise SystemExit(0)
session = _oshot.run_500m_campaign(SESSION)
print('CYMEK 500M STATUS:', session['status'])
print('tokens:', session.get('tokens_done'), '/', '500M')
from google.colab import files
if session.get('campaign_bundle'):
    files.download(session['campaign_bundle'])
    print('CAMPAIGN BUNDLE downloaded')